In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv
/kaggle/input/bilstm_lm/other/default/1/bilstm_doc_best.pt
/kaggle/input/bpe-artifacts-v1/bpe_code.vocab.json
/kaggle/input/bpe-artifacts-v1/bpe_code.meta.json
/kaggle/input/bpe-artifacts-v1/bpe_doc.vocab.json
/kaggle/input/bpe-artifacts-v1/bpe_doc.merges
/kaggle/input/bpe-artifacts-v1/bpe_code.merges
/kaggle/input/bpe-artifacts-v1/bpe_doc.meta.json
/kaggle/input/word-to-vec/w2v_code.model
/kaggle/input/word-to-vec/w2v_combined.model
/kaggle/input/word-to-vec/w2v_doc.model


In [4]:
# ======================
# Cell 1: Imports & Setup
# ======================
import os, json, pickle
import torch
import torch.nn as nn
from typing import List, Dict, Any, Optional

# Force GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True   # optimize for fixed input sizes

print("Using device:", DEVICE)


Using device: cuda


In [5]:
# ================================
# Cell 2: Minimal Tokenizer + BiLSTM
# ================================
class WhitespaceTokenizer:
    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = {}
        self.pad_token, self.unk_token = "<PAD>", "<UNK>"
        self.bos_token, self.eos_token = "<BOS>", "<EOS>"
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.token_to_id = {
            self.pad_token: self.pad_id,
            self.unk_token: self.unk_id,
            self.bos_token: self.bos_id,
            self.eos_token: self.eos_id,
        }
        self.id_to_token = {i:t for t,i in self.token_to_id.items()}

    def encode(self, text: str):
        toks = text.strip().split()
        return [self.bos_id] + [self.token_to_id.get(t, self.unk_id) for t in toks] + [self.eos_id]

    def decode(self, ids):
        toks = []
        for i in ids:
            if i in (self.pad_id, self.bos_id, self.eos_id): continue
            toks.append(self.id_to_token.get(i, self.unk_token))
        return " ".join(toks)

class BiLSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.3, tie_weights=False):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, vocab_size)
    def forward(self, x, lengths):
        emb = self.embed(x)
        out, _ = self.lstm(emb)
        logits = self.fc(out)
        return logits


In [9]:
def load_bilstm(ckpt_path: str) -> Dict[str, Any]:
    state = torch.load(ckpt_path, map_location=DEVICE)
    print("Checkpoint keys:", list(state.keys()))  # debug

    # No vocab saved → use default tokenizer (special tokens only)
    tokenizer = WhitespaceTokenizer()

    # Guess model size (you may need to adjust if mismatch occurs)
    vocab_size = 5000  # 🔴 set a reasonable size (from BPE joint vocab size)
    embed_dim = 256
    hidden_dim = 512
    num_layers = 2

    model = BiLSTMLanguageModel(
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=0.3,
    ).to(DEVICE)

    # Load weights directly
    model.load_state_dict(state, strict=False)
    model.eval()
    return {"model": model, "tokenizer": tokenizer}


In [10]:
# ================================
# Cell 4: Generation Utilities
# ================================
@torch.no_grad()
def greedy_generate(model, tokenizer, prompt: str, max_new_tokens=80) -> str:
    ids = tokenizer.encode(prompt)
    inp = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    for _ in range(max_new_tokens):
        lengths = torch.tensor([inp.size(1)], dtype=torch.long, device=DEVICE)
        logits = model(inp, lengths)
        next_id = int(torch.argmax(logits[0, -1]))
        inp = torch.cat([inp, torch.tensor([[next_id]], device=DEVICE)], dim=1)
        if tokenizer.id_to_token.get(next_id) == tokenizer.eos_token:
            break
    return tokenizer.decode(inp[0].tolist())

def build_context_hint(code_text: str, w2v: Optional[Any], top_k=5) -> str:
    if w2v is None: return ""
    tokens = code_text.split()
    hints = []
    for t in tokens[:10]:
        if t in w2v.wv:
            sims = w2v.wv.most_similar(t, topn=2)
            hints.extend([s for s,_ in sims])
    return " Context: " + " ".join(hints[:top_k]) if hints else ""

def generate_for_function(code: str, name: str,
                          bilstm, bpe_models, w2v,
                          max_tokens=120) -> Dict[str, str]:
    # Fallback tokenizer = whitespace + your BPE vocab
    code_tokens = code.strip().split()[:100]
    base_prompt = f"Summarize function {name or ''}: " + " ".join(code_tokens)
    context = build_context_hint(" ".join(code_tokens), w2v)
    summary = greedy_generate(bilstm["model"], bilstm["tokenizer"],
                              base_prompt + context, max_new_tokens=max_tokens//2)
    doc_prompt = f"Detailed docstring for {name or 'function'}: " + summary
    docstring = greedy_generate(bilstm["model"], bilstm["tokenizer"],
                                doc_prompt, max_new_tokens=max_tokens)
    return {"summary": summary, "docstring": docstring}


In [17]:
# # ================================
# # Cell 3: Load BiLSTM + BPE vocab + Word2Vec (GPU)
# # ================================
# import json, re
# from typing import Any, Optional

# # ---- Paths (from your screenshot/names)
# BILSTM_CKPT = "/kaggle/input/bilstm_lm/other/default/1/bilstm_doc_best.pt"
# BPE_DIR     = "/kaggle/input/bpe-artifacts-v1"
# W2V_PATH    = "/kaggle/input/word-to-vec/w2v_combined.model"

# # ---- Helper: read BPE vocab (handles dict or list)
# def _read_bpe_vocab(path: str) -> list[str]:
#     if not os.path.exists(path):
#         return []
#     with open(path, "r", encoding="utf-8") as f:
#         data = json.load(f)
#     if isinstance(data, dict):
#         # assume token -> id mapping; order by id
#         try:
#             tokens_sorted = sorted(data.items(), key=lambda kv: kv[1])
#             return [t for t, _ in tokens_sorted]
#         except Exception:
#             # fallback: stable key order
#             return list(data.keys())
#     elif isinstance(data, list):
#         return data
#     else:
#         return []

# # ---- Build combined vocab list from your BPE artifacts
# doc_tokens  = _read_bpe_vocab(os.path.join(BPE_DIR, "bpe_doc.vocab.json"))
# code_tokens = _read_bpe_vocab(os.path.join(BPE_DIR, "bpe_code.vocab.json"))

# # merge (doc first, then code uniques)
# combined = []
# seen = set()
# for t in doc_tokens + code_tokens:
#     if t not in seen:
#         combined.append(t); seen.add(t)

# # ---- Load raw state_dict and infer model sizes from weights (no config/vocab saved)
# state = torch.load(BILSTM_CKPT, map_location=DEVICE)
# print("Checkpoint keys:", list(state.keys())[:6], "... (total:", len(state), ")")

# # infer sizes
# vocab_size_in_ckpt = state["embedding.weight"].shape[0]
# embed_dim_in_ckpt  = state["embedding.weight"].shape[1]
# hidden2_in_fc      = state["fc.weight"].shape[1]              # = hidden_dim * 2
# hidden_dim_in_ckpt = hidden2_in_fc // 2

# # count num_layers by scanning keys like lstm.weight_ih_l0, l1, ...
# layer_ids = set()
# pat = re.compile(r"lstm\.weight_ih_l(\d+)(?:_reverse)?$")
# for k in state.keys():
#     m = pat.match(k)
#     if m:
#         layer_ids.add(int(m.group(1)))
# num_layers_in_ckpt = (max(layer_ids) + 1) if layer_ids else 2

# print(f"Inferred: vocab_size={vocab_size_in_ckpt}, embed_dim={embed_dim_in_ckpt}, hidden_dim={hidden_dim_in_ckpt}, num_layers={num_layers_in_ckpt}")

# # ---- Build tokenizer mapping to EXACTLY match vocab_size from checkpoint
# # Special tokens occupy 0..3
# SPECIALS = ["<PAD>","<UNK>","<BOS>","<EOS>"]
# needed = vocab_size_in_ckpt - len(SPECIALS)

# # Pad or trim combined list to 'needed'
# if len(combined) < needed:
#     # add synthetic tokens to fill the gap
#     for i in range(needed - len(combined)):
#         combined.append(f"<EXTRA_{i}>")
# elif len(combined) > needed:
#     combined = combined[:needed]

# # Construct mappings
# token_to_id = {
#     "<PAD>":0, "<UNK>":1, "<BOS>":2, "<EOS>":3,
#     **{tok:(i+4) for i, tok in enumerate(combined)}
# }
# id_to_token = {i:t for t,i in token_to_id.items()}

# # Install into our tokenizer instance
# tokenizer = WhitespaceTokenizer()
# tokenizer.token_to_id = token_to_id
# tokenizer.id_to_token = id_to_token

# # ---- Build model with EXACT inferred shapes, then load raw state dict
# model = BiLSTMLanguageModel(
#     vocab_size=vocab_size_in_ckpt,
#     embed_dim=embed_dim_in_ckpt,
#     hidden_dim=hidden_dim_in_ckpt,
#     num_layers=num_layers_in_ckpt,
#     dropout=0.0,     # dropout irrelevant at inference
# ).to(DEVICE)

# missing, unexpected = model.load_state_dict(state, strict=False)
# print("Loaded weights. Missing keys:", missing, "| Unexpected keys:", unexpected)
# model.eval()

# # Wrap for downstream usage
# bilstm = {"model": model, "tokenizer": tokenizer}
# print("✅ BiLSTM ready on", DEVICE, "with vocab:", len(tokenizer.token_to_id))

# # ---- Load Word2Vec (optional)
# try:
#     from gensim.models import Word2Vec
#     GENSIM_AVAILABLE = True
# except ImportError:
#     GENSIM_AVAILABLE = False

# def load_word2vec(path: str) -> Optional[Any]:
#     if not (GENSIM_AVAILABLE and os.path.exists(path)):
#         return None
#     return Word2Vec.load(path)

# w2v = load_word2vec(W2V_PATH)
# print("✅ Word2Vec loaded:", bool(w2v))
# ================================
# Cell 3 (Rectified): Load BiLSTM + Build Combined BPE Vocab + Word2Vec
# ================================
import json, re
from typing import Any, Optional

# ---- Paths
BILSTM_CKPT = "/kaggle/input/bilstm_lm/other/default/1/bilstm_doc_best.pt"
BPE_DIR     = "/kaggle/input/bpe-artifacts-v1"
W2V_PATH    = "/kaggle/input/word-to-vec/w2v_combined.model"

# ---- Helper: load vocab JSON (dict or list)
def _read_vocab(path: str):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        # some BPE libs save token → freq or token → id
        try:
            return [t for t, _ in sorted(data.items(), key=lambda kv: kv[1])]
        except Exception:
            return list(data.keys())
    elif isinstance(data, list):
        return data
    return []

# ---- Load code + doc vocabs
code_tokens = _read_vocab(os.path.join(BPE_DIR, "bpe_code.vocab.json"))
doc_tokens  = _read_vocab(os.path.join(BPE_DIR, "bpe_doc.vocab.json"))

# ---- Merge unique tokens
combined_tokens, seen = [], set()
for tok in doc_tokens + code_tokens:
    if tok not in seen:
        combined_tokens.append(tok)
        seen.add(tok)

print("Raw merged vocab size:", len(combined_tokens))

# ---- Load checkpoint (state_dict only)
state = torch.load(BILSTM_CKPT, map_location=DEVICE)
print("Checkpoint keys:", list(state.keys())[:6], "...")

# ---- Infer model config
vocab_size_in_ckpt = state["embedding.weight"].shape[0]
embed_dim_in_ckpt  = state["embedding.weight"].shape[1]
hidden_dim_in_ckpt = state["fc.weight"].shape[1] // 2

layer_ids = set()
pat = re.compile(r"lstm\.weight_ih_l(\d+)(?:_reverse)?$")
for k in state.keys():
    m = pat.match(k)
    if m: layer_ids.add(int(m.group(1)))
num_layers_in_ckpt = (max(layer_ids) + 1) if layer_ids else 2

print(f"Inferred: vocab={vocab_size_in_ckpt}, embed_dim={embed_dim_in_ckpt}, hidden_dim={hidden_dim_in_ckpt}, layers={num_layers_in_ckpt}")

# ---- Build tokenizer aligned with checkpoint
SPECIALS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
needed = vocab_size_in_ckpt - len(SPECIALS)

if len(combined_tokens) < needed:
    for i in range(needed - len(combined_tokens)):
        combined_tokens.append(f"<EXTRA_{i}>")
elif len(combined_tokens) > needed:
    combined_tokens = combined_tokens[:needed]

token_to_id = {t: i for i, t in enumerate(SPECIALS)}
for i, tok in enumerate(combined_tokens, start=len(SPECIALS)):
    token_to_id[tok] = i
id_to_token = {i: t for t, i in token_to_id.items()}

# ---- Install into tokenizer
tokenizer = WhitespaceTokenizer()
tokenizer.token_to_id = token_to_id
tokenizer.id_to_token = id_to_token

print("Final vocab size (aligned):", len(token_to_id))

# ---- Build model with inferred sizes
model = BiLSTMLanguageModel(
    vocab_size=vocab_size_in_ckpt,
    embed_dim=embed_dim_in_ckpt,
    hidden_dim=hidden_dim_in_ckpt,
    num_layers=num_layers_in_ckpt,
    dropout=0.0
).to(DEVICE)

model.load_state_dict(state, strict=False)
model.eval()

bilstm = {"model": model, "tokenizer": tokenizer}
print("✅ BiLSTM loaded on", DEVICE)

# ---- Load Word2Vec
try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False

def load_word2vec(path: str) -> Optional[Any]:
    if not (GENSIM_AVAILABLE and os.path.exists(path)):
        return None
    return Word2Vec.load(path)

w2v = load_word2vec(W2V_PATH)
print("✅ Word2Vec loaded:", bool(w2v))


Raw merged vocab size: 511
Checkpoint keys: ['embedding.weight', 'lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0', 'lstm.weight_ih_l0_reverse'] ...
Inferred: vocab=4919, embed_dim=100, hidden_dim=512, layers=2
Final vocab size (aligned): 4915
✅ BiLSTM loaded on cuda
✅ Word2Vec loaded: True


In [18]:
# ================================
# Cell 4: Generation Utilities (BPE-friendly)
# ================================
@torch.no_grad()
def greedy_generate(model: nn.Module, tok: WhitespaceTokenizer, prompt: str, max_new_tokens: int = 80) -> str:
    ids = tok.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    out_ids = ids[:]  # track full sequence
    for _ in range(max_new_tokens):
        lengths = torch.tensor([x.size(1)], dtype=torch.long, device=DEVICE)
        logits = model(x, lengths)
        next_id = int(torch.argmax(logits[0, -1]))
        out_ids.append(next_id)
        x = torch.cat([x, torch.tensor([[next_id]], device=DEVICE)], dim=1)
        if tok.id_to_token.get(next_id, "") == "<EOS>":
            break
    # instead of `<UNK>`, fallback to raw id or token string
    return " ".join([tok.id_to_token.get(i, f"<ID{i}>") for i in out_ids])

def generate_for_function(code: str, name: str,
                          bilstm_bundle: dict, max_tokens: int = 120) -> dict:
    model = bilstm_bundle["model"]
    tok   = bilstm_bundle["tokenizer"]

    code_tokens = code.strip().split()[:100]
    base_prompt = f"Summarize function {name or ''}: " + " ".join(code_tokens)
    summary = greedy_generate(model, tok, base_prompt, max_new_tokens=max_tokens//2)

    doc_prompt = f"Detailed docstring for {name or 'function'}: " + summary
    docstring = greedy_generate(model, tok, doc_prompt, max_new_tokens=max_tokens)

    return {"summary": summary, "docstring": docstring}


In [19]:
# ================================
# Cell 5: Test again with fallback decode
# ================================
sample_code = """
def multiply(x, y):
    return x * y
"""
out = generate_for_function(sample_code, "multiply", bilstm, max_tokens=60)
print("Summary:", out["summary"])
print("Docstring:", out["docstring"])


Summary: <ID2> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> return x * y <ID3> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1>
Docstring: <ID2> <ID1> <ID1> for <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> return x * y <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID3> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1> <ID1>


In [23]:
# app.py
import os, re, json
from typing import Dict, Any, Optional, List
import torch
import torch.nn as nn
import gradio as gr

# --------------------
# Dummy minimal loader
# (replace with your real BiLSTM load code)
# --------------------
class WhitespaceTokenizer:
    def __init__(self):
        self.pad_token, self.unk_token = "<PAD>", "<UNK>"
        self.bos_token, self.eos_token = "<BOS>", "<EOS>"
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.token_to_id = {self.pad_token:0, self.unk_token:1,
                            self.bos_token:2, self.eos_token:3}
        self.id_to_token = {i:t for t,i in self.token_to_id.items()}
    def encode(self, text: str):
        return [self.bos_id] + [self.token_to_id.get(t, self.unk_id) for t in text.split()] + [self.eos_id]
    def decode(self, ids: List[int]) -> str:
        return " ".join([self.id_to_token.get(i, f"<ID{i}>") for i in ids if i not in (self.pad_id,self.bos_id,self.eos_id)])

class BiLSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, bidirectional=True)
        self.fc    = nn.Linear(hidden_dim*2, vocab_size)
    def forward(self, x, lengths):
        emb = self.embed(x)
        out, _ = self.lstm(emb)
        return self.fc(out)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fake model/tokenizer for demo
tokenizer = WhitespaceTokenizer()
model = BiLSTMLanguageModel(vocab_size=len(tokenizer.token_to_id)).to(DEVICE)
bundle = {"model": model, "tokenizer": tokenizer}

@torch.no_grad()
def greedy_generate(model, tok, prompt, max_new_tokens=40):
    ids = tok.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    out_ids = ids[:]
    for _ in range(max_new_tokens):
        logits = model(x, torch.tensor([x.size(1)], device=DEVICE))
        next_id = int(torch.argmax(logits[0,-1]))
        out_ids.append(next_id)
        x = torch.cat([x, torch.tensor([[next_id]], device=DEVICE)], dim=1)
        if tok.id_to_token.get(next_id) == "<EOS>":
            break
    return tok.decode(out_ids)

def generate_for_function(code: str, name: str):
    prompt = f"Summarize function {name or ''}: {code}"
    summary = greedy_generate(bundle["model"], bundle["tokenizer"], prompt)
    doc_prompt = f"Detailed docstring for {name or 'function'}: {summary}"
    docstring = greedy_generate(bundle["model"], bundle["tokenizer"], doc_prompt)
    return summary, docstring

# --------------------
# Enhanced Gen-Z Styling
# --------------------
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;800&family=Inter:wght@300;400;600;700&display=swap');

* {
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
}

body {
    background: linear-gradient(135deg, #0c0c0c 0%, #1a0a2e 25%, #16213e 50%, #0f0f23 75%, #000000 100%);
    background-size: 400% 400%;
    animation: gradientShift 15s ease infinite;
    font-family: 'Inter', sans-serif;
    color: #ffffff;
    overflow-x: hidden;
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

.gradio-container {
    background: rgba(15, 15, 35, 0.95) !important;
    backdrop-filter: blur(20px) !important;
    border: 1px solid rgba(139, 69, 19, 0.3) !important;
    border-radius: 24px !important;
    box-shadow: 
        0 25px 50px -12px rgba(0, 0, 0, 0.8),
        0 0 0 1px rgba(255, 255, 255, 0.05),
        inset 0 1px 0 rgba(255, 255, 255, 0.1) !important;
    max-width: 1200px !important;
    margin: 2rem auto !important;
    padding: 2rem !important;
}

.gradio-container::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    bottom: 0;
    background: linear-gradient(45deg, rgba(255,20,147,0.1) 0%, rgba(0,191,255,0.1) 50%, rgba(255,20,147,0.1) 100%);
    border-radius: 24px;
    z-index: -1;
    opacity: 0.7;
    animation: pulse 4s ease-in-out infinite alternate;
}

@keyframes pulse {
    0% { opacity: 0.3; }
    100% { opacity: 0.7; }
}

/* Modern Button Styling */
button {
    background: linear-gradient(135deg, #ff1493 0%, #00bfff 50%, #ff1493 100%) !important;
    background-size: 200% 200% !important;
    border: none !important;
    color: white !important;
    font-weight: 600 !important;
    font-family: 'Inter', sans-serif !important;
    border-radius: 16px !important;
    padding: 0.875rem 2rem !important;
    font-size: 1rem !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
    box-shadow: 
        0 10px 25px rgba(255, 20, 147, 0.4),
        0 5px 10px rgba(0, 0, 0, 0.2),
        inset 0 1px 0 rgba(255, 255, 255, 0.2) !important;
    position: relative !important;
    overflow: hidden !important;
}

button::before {
    content: '';
    position: absolute;
    top: 0;
    left: -100%;
    width: 100%;
    height: 100%;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.2), transparent);
    transition: left 0.5s;
}

button:hover {
    background-position: 100% 0% !important;
    box-shadow: 
        0 15px 35px rgba(255, 20, 147, 0.6),
        0 8px 15px rgba(0, 0, 0, 0.3),
        inset 0 1px 0 rgba(255, 255, 255, 0.3) !important;
    transform: translateY(-2px) scale(1.02) !important;
}

button:hover::before {
    left: 100%;
}

button:active {
    transform: translateY(0px) scale(0.98) !important;
}

/* Input Field Styling */
textarea, input[type="text"] {
    background: rgba(20, 20, 40, 0.9) !important;
    color: #ffffff !important;
    border: 1px solid rgba(255, 20, 147, 0.3) !important;
    border-radius: 12px !important;
    padding: 1rem !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.9rem !important;
    line-height: 1.6 !important;
    box-shadow: 
        inset 0 2px 4px rgba(0, 0, 0, 0.3),
        0 1px 0 rgba(255, 255, 255, 0.05) !important;
    backdrop-filter: blur(10px) !important;
}

textarea:focus, input[type="text"]:focus {
    border-color: rgba(0, 191, 255, 0.6) !important;
    box-shadow: 
        inset 0 2px 4px rgba(0, 0, 0, 0.3),
        0 0 0 3px rgba(0, 191, 255, 0.2),
        0 1px 0 rgba(255, 255, 255, 0.05) !important;
    outline: none !important;
}

/* Label Styling */
.gr-textbox label, .gr-button label {
    color: #ffffff !important;
    font-weight: 600 !important;
    font-size: 1rem !important;
    margin-bottom: 0.5rem !important;
    text-shadow: 0 2px 4px rgba(0, 0, 0, 0.5) !important;
}

/* Row and Column Enhancements */
.gr-row {
    gap: 1.5rem !important;
    margin: 1rem 0 !important;
}

/* Glass morphism effect for output areas */
.gr-textbox {
    background: rgba(30, 30, 60, 0.8) !important;
    backdrop-filter: blur(15px) !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 12px !important;
}

/* Special styling for output textboxes */
#summary-output textarea, #docstring-output textarea {
    background: rgba(25, 45, 65, 0.9) !important;
    border: 1px solid rgba(0, 191, 255, 0.2) !important;
    color: #e0e0e0 !important;
}

#summary-output textarea:focus, #docstring-output textarea:focus {
    border-color: rgba(255, 20, 147, 0.5) !important;
    box-shadow: 
        inset 0 2px 4px rgba(0, 0, 0, 0.3),
        0 0 0 3px rgba(255, 20, 147, 0.2),
        0 1px 0 rgba(255, 255, 255, 0.05) !important;
}

/* Placeholder text styling */
::placeholder {
    color: rgba(255, 255, 255, 0.5) !important;
    font-style: italic !important;
}

/* Scrollbar styling */
::-webkit-scrollbar {
    width: 8px;
    height: 8px;
}

::-webkit-scrollbar-track {
    background: rgba(20, 20, 40, 0.5);
    border-radius: 4px;
}

::-webkit-scrollbar-thumb {
    background: linear-gradient(45deg, #ff1493, #00bfff);
    border-radius: 4px;
}

::-webkit-scrollbar-thumb:hover {
    background: linear-gradient(45deg, #00bfff, #ff1493);
}

/* Header glow effect */
h1 {
    background: linear-gradient(135deg, #ff1493, #00bfff, #ff1493);
    background-size: 200% 200%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    animation: gradientText 3s ease infinite;
    text-shadow: 0 0 30px rgba(255, 20, 147, 0.5);
    font-family: 'Inter', sans-serif;
    font-weight: 800;
    font-size: 2.5rem;
    margin-bottom: 2rem;
}

@keyframes gradientText {
    0%, 100% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
}

/* Card-like sections */
.gr-group {
    background: rgba(25, 25, 50, 0.6) !important;
    border-radius: 16px !important;
    padding: 1.5rem !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    backdrop-filter: blur(10px) !important;
}

/* Subtle animations */
.gr-textbox, .gr-button {
    animation: fadeInUp 0.6s ease-out;
}

@keyframes fadeInUp {
    from {
        opacity: 0;
        transform: translateY(20px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

/* Mobile responsiveness */
@media (max-width: 768px) {
    .gradio-container {
        margin: 1rem !important;
        padding: 1rem !important;
        border-radius: 16px !important;
    }
    
    h1 {
        font-size: 2rem !important;
    }
    
    button {
        padding: 0.75rem 1.5rem !important;
        font-size: 0.9rem !important;
    }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:
    gr.HTML("""
    <div style='text-align: center; margin-bottom: 2rem;'>
        <h1>🚀 AI Doc Generator 🔥</h1>
        <p style='color: rgba(255,255,255,0.8); font-size: 1.1rem; margin-top: -1rem;'>
            Transform your code into beautiful documentation with AI magic ✨
        </p>
    </div>
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            code_in = gr.Textbox(
                label="📝 Function Code", 
                lines=12, 
                placeholder="def my_awesome_function(x, y):\n    \"\"\"Paste your Python function here...\"\"\"\n    return x + y",
                elem_id="code-input"
            )
        with gr.Column(scale=1):
            name_in = gr.Textbox(
                label="🏷️ Function Name", 
                placeholder="my_awesome_function",
                elem_id="name-input"
            )
            btn = gr.Button("✨ Generate Magic", elem_id="generate-btn")
    
    with gr.Row():
        with gr.Column():
            summary_out = gr.Textbox(
                label="📋 AI Summary", 
                lines=3,
                elem_id="summary-output",
                interactive=True,
                placeholder="AI-generated summary will appear here... or type your own!"
            )
        with gr.Column():
            doc_out = gr.Textbox(
                label="📚 Generated Docstring", 
                lines=8,
                elem_id="docstring-output",
                interactive=True,
                placeholder="AI-generated docstring will appear here... or write your own!"
            )
    
    btn.click(generate_for_function, inputs=[code_in, name_in], outputs=[summary_out, doc_out])

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://4781aa31b5323ec15f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
